# Survey Results Summary
Loads the output of `survey_planets.py` and provides statistics to guide sector/planet selection.

**Run order:** Cell 1 → Cell 2 → any of Cells 3–7 in any order.

In [ ]:
# Cell 1 — Setup
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display

RESULTS_DIR  = 'results'
SUMMARY_CSV  = os.path.join(RESULTS_DIR, 'survey_summary.csv')

# ── Load master summary ───────────────────────────────────────────────────────
df_raw = pd.read_csv(SUMMARY_CSV)

# Separate the Total row from planet rows
df_total   = df_raw[df_raw['System'] == 'Total']
df_summary = df_raw[(df_raw['System'] != 'Total') & (df_raw['status'] == 'ok')].copy()
df_errors  = df_raw[(df_raw['System'] != 'Total') & (df_raw['status'] != 'ok')].copy()

df_summary = df_summary.sort_values('total_usable', ascending=False).reset_index(drop=True)

# ── Load per-planet sector reports ───────────────────────────────────────────
sector_reports = {}
for name in df_summary['System']:
    path = os.path.join(RESULTS_DIR, name, 'sector_report.csv')
    if os.path.exists(path):
        sr = pd.read_csv(path)
        sr = sr[sr['sector'] != 'Total'].copy()
        sr['sector'] = sr['sector'].astype(int)
        sector_reports[name] = sr

# ── Build planet × sector matrix from per-planet reports ─────────────────────
all_sectors = sorted(set(
    int(s)
    for sr in sector_reports.values()
    for s in sr['sector']
))

matrix_rows = []
for name in df_summary['System']:
    sr = sector_reports.get(name)
    row = {'System': name}
    if sr is not None:
        sec_map = dict(zip(sr['sector'], sr['n_usable']))
        for s in all_sectors:
            row[s] = sec_map.get(s, 0)
    else:
        for s in all_sectors:
            row[s] = 0
    matrix_rows.append(row)

matrix = pd.DataFrame(matrix_rows).set_index('System')

# ── Sector-level aggregates ───────────────────────────────────────────────────
sec_planet_count  = (matrix > 0).sum()          # planets with ≥1 usable transit per sector
sec_transit_total = matrix.sum()                 # total usable transits per sector
df_sectors = pd.DataFrame({
    'sector':          all_sectors,
    'n_planets':       sec_planet_count.values,
    'total_usable':    sec_transit_total.values,
}).sort_values('n_planets', ascending=False).reset_index(drop=True)

grand_total = int(df_summary['total_usable'].sum())

print(f'Planets (ok):       {len(df_summary)}')
print(f'Planets (error):    {len(df_errors)}')
print(f'Sectors covered:    {len(all_sectors)}')
print(f'Grand total usable: {grand_total} transits')
if len(df_errors):
    print(f'\nErrors: {df_errors["System"].tolist()}')

In [ ]:
# Cell 2 — Planet ranking by total usable transits
fig, ax = plt.subplots(figsize=(10, max(5, len(df_summary) * 0.32)))

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.85, len(df_summary))[::-1])
bars = ax.barh(df_summary['System'][::-1],
               df_summary['total_usable'][::-1],
               color=colors, edgecolor='white', linewidth=0.4)

# Annotate bars
for bar, val in zip(bars, df_summary['total_usable'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(int(val)), va='center', fontsize=7.5)

ax.set_xlabel('Total usable transits (all sectors combined)', fontsize=10)
ax.set_title('Priority Planets — Total Usable Transits\n'
             '(pre + in-transit + post windows all ≥ min_pts)', fontsize=11, fontweight='bold')
ax.axvline(df_summary['total_usable'].median(), color='navy', lw=1.2, ls='--',
           label=f'Median = {df_summary["total_usable"].median():.0f}')
ax.legend(fontsize=9)
ax.set_xlim(0, df_summary['total_usable'].max() * 1.12)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'plot_planet_ranking.png'), dpi=130, bbox_inches='tight')
plt.show()

# Table
tbl = df_summary[['System','n_sectors','total_usable']].copy()
tbl.index = range(1, len(tbl)+1)
tbl.columns = ['Planet', 'Sectors', 'Usable transits']
display(tbl)

In [ ]:
# Cell 2b — Per-planet usable transit summary table (sector-by-sector breakdown)
# Shows each planet's usable transits per sector, with a Total column.

# Build readable table: planets as rows, only sectors where at least one planet has data
active_sectors = sorted([s for s in all_sectors if matrix[s].sum() > 0])
tbl_data = matrix[active_sectors].copy()
tbl_data['Total'] = tbl_data.sum(axis=1)
tbl_data = tbl_data.sort_values('Total', ascending=False)

# Rename sector columns to S## format
tbl_data.columns = [f'S{s:02d}' if isinstance(s, int) else s for s in tbl_data.columns]

# Replace zeros with blank for readability
tbl_display = tbl_data.replace(0, '')

print(f'Total usable transits per planet across all sectors')
print(f'(blank = 0 usable transits in that sector)\n')
display(tbl_display)

# Also print a compact text summary sorted by total
print('\n── Compact summary ──')
print(f'{"Planet":<25} {"Sectors":>8} {"Total":>7}')
print('─' * 44)
for name, row in tbl_data.iterrows():
    n_active = int((row[:-1] > 0).sum())
    total    = int(row['Total'])
    print(f'{name:<25} {n_active:>8} {total:>7}')
print('─' * 44)
print(f'{"GRAND TOTAL":<25} {"":>8} {int(tbl_data["Total"].sum()):>7}')

In [ ]:
# Cell 2c — Stacked bar: usable transits per planet, broken down by sector
# Each bar = one planet; each colored segment = one sector's contribution.
# Only sectors with at least 1 usable transit across any planet are shown.

# Use top contributing sectors for the legend (too many sectors = cluttered)
TOP_SECTORS_IN_LEGEND = 15   # show only the top N sectors by total transits in legend

sector_totals = matrix[active_sectors].sum().sort_values(ascending=False)
top_sec = list(sector_totals.head(TOP_SECTORS_IN_LEGEND).index)
other_sec = [s for s in active_sectors if s not in top_sec]

# Colors: one per top sector
cmap_sec = plt.cm.tab20
sec_colors = {s: cmap_sec(i / len(top_sec)) for i, s in enumerate(top_sec)}
other_color = '#cccccc'

planets_ordered = tbl_data.index.tolist()   # already sorted by Total desc
totals = tbl_data['Total'].values

fig, ax = plt.subplots(figsize=(12, max(5, len(planets_ordered) * 0.34)))

bottoms = np.zeros(len(planets_ordered))

for sec in top_sec:
    col = f'S{sec:02d}'
    vals = tbl_data[col].replace('', 0).astype(int).values
    ax.barh(planets_ordered[::-1], vals[::-1],
            left=bottoms[::-1],
            color=sec_colors[sec], edgecolor='white', linewidth=0.3,
            label=f'S{sec:02d}')
    bottoms += vals

# Lump remaining sectors into "Other"
if other_sec:
    other_cols = [f'S{s:02d}' for s in other_sec]
    other_vals = tbl_data[other_cols].replace('', 0).astype(int).sum(axis=1).values
    ax.barh(planets_ordered[::-1], other_vals[::-1],
            left=bottoms[::-1],
            color=other_color, edgecolor='white', linewidth=0.3,
            label='Other sectors')

# Annotate total at end of each bar
for i, (planet, total) in enumerate(zip(planets_ordered[::-1], totals[::-1])):
    if total > 0:
        ax.text(total + 0.3, i, str(int(total)), va='center', fontsize=7.5, color='#222')

ax.set_xlabel('Usable transits', fontsize=10)
ax.set_title('Usable Transits per Planet — Sector Breakdown\n'
             '(each color segment = one TESS sector)', fontsize=11, fontweight='bold')
ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left', title='Sector')
ax.set_xlim(0, totals.max() * 1.12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'plot_planet_sector_breakdown.png'),
            dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 3 — Sector ranking: planets per sector and total usable transits
top_n = 30
df_sec_top = df_sectors.head(top_n)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(5, top_n * 0.32)))

sec_labels = [f'S{int(s):02d}' for s in df_sec_top['sector']]

# Left: number of planets
c1 = plt.cm.Blues(np.linspace(0.35, 0.85, top_n))
bars1 = ax1.barh(sec_labels[::-1], df_sec_top['n_planets'][::-1],
                 color=c1, edgecolor='white', linewidth=0.4)
for bar, val in zip(bars1, df_sec_top['n_planets'][::-1]):
    ax1.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
             str(int(val)), va='center', fontsize=7.5)
ax1.set_xlabel('Number of planets with ≥1 usable transit', fontsize=9)
ax1.set_title('Planets per Sector', fontsize=10, fontweight='bold')
ax1.set_xlim(0, df_sec_top['n_planets'].max() * 1.15)
ax1.spines[['top','right']].set_visible(False)

# Right: total usable transits
c2 = plt.cm.Oranges(np.linspace(0.35, 0.85, top_n))
bars2 = ax2.barh(sec_labels[::-1], df_sec_top['total_usable'][::-1],
                 color=c2, edgecolor='white', linewidth=0.4)
for bar, val in zip(bars2, df_sec_top['total_usable'][::-1]):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             str(int(val)), va='center', fontsize=7.5)
ax2.set_xlabel('Total usable transits across all planets', fontsize=9)
ax2.set_title('Total Usable Transits per Sector', fontsize=10, fontweight='bold')
ax2.set_xlim(0, df_sec_top['total_usable'].max() * 1.15)
ax2.spines[['top','right']].set_visible(False)

fig.suptitle(f'Top {top_n} Sectors by Planet Coverage', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'plot_sector_ranking.png'), dpi=130, bbox_inches='tight')
plt.show()

# Table
tbl2 = df_sectors.head(top_n).copy()
tbl2['sector'] = tbl2['sector'].apply(lambda s: f'S{int(s):02d}')
tbl2.index = range(1, len(tbl2)+1)
tbl2.columns = ['Sector', 'Planets', 'Total usable transits']
display(tbl2)

In [ ]:
# Cell 4 — Planet × sector heatmap
# Rows = planets (sorted by total usable desc)
# Columns = sectors with at least 1 usable transit across all planets

# Trim to sectors that have any usable transits
active_sectors = [s for s in all_sectors if matrix[s].sum() > 0]
mat = matrix[active_sectors]

fig, ax = plt.subplots(figsize=(max(14, len(active_sectors) * 0.22),
                                 max(6,  len(df_summary) * 0.35)))

cmap = plt.cm.YlOrRd
cmap.set_under('#f0f0f0')   # zero = light grey
vmax = mat.values.max()

im = ax.imshow(mat.values, aspect='auto', cmap=cmap,
               vmin=0.5, vmax=vmax, interpolation='nearest')

# Annotate non-zero cells
for r in range(mat.shape[0]):
    for c in range(mat.shape[1]):
        v = int(mat.values[r, c])
        if v > 0:
            ax.text(c, r, str(v), ha='center', va='center',
                    fontsize=6, color='black' if v < vmax*0.6 else 'white')

ax.set_xticks(range(len(active_sectors)))
ax.set_xticklabels([f'S{s:02d}' for s in active_sectors],
                   rotation=90, fontsize=7)
ax.set_yticks(range(len(mat)))
ax.set_yticklabels(mat.index, fontsize=8)
ax.set_xlabel('TESS Sector', fontsize=10)
ax.set_title('Usable Transits per Planet per Sector\n'
             '(grey = no usable transits)', fontsize=11, fontweight='bold')

cb = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.01)
cb.set_label('Usable transits', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'plot_heatmap.png'), dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 5 — Sector detail: for a chosen sector, show all planets and their usable transit counts
# Edit FOCUS_SECTORS to inspect specific sectors

FOCUS_SECTORS = [75, 41, 40, 74, 81]   # change these to sectors you are considering

for sec in FOCUS_SECTORS:
    col = sec
    if col not in matrix.columns:
        print(f'S{sec:02d}: not found in data')
        continue

    sub = matrix[[col]].copy()
    sub.columns = ['usable']
    sub = sub[sub['usable'] > 0].sort_values('usable', ascending=False)

    print(f'\n── Sector {sec:02d}  |  {len(sub)} planets  |  {int(sub["usable"].sum())} total usable ──')

    fig, ax = plt.subplots(figsize=(7, max(3, len(sub) * 0.38)))
    colors = plt.cm.RdYlGn(np.linspace(0.25, 0.85, len(sub))[::-1])
    bars = ax.barh(sub.index[::-1], sub['usable'][::-1],
                   color=colors, edgecolor='white', linewidth=0.4)
    for bar, val in zip(bars, sub['usable'][::-1]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                str(int(val)), va='center', fontsize=8)
    ax.set_xlabel('Usable transits', fontsize=9)
    ax.set_title(f'Sector {sec:02d} — {len(sub)} planets  ({int(sub["usable"].sum())} transits)',
                 fontsize=10, fontweight='bold')
    ax.set_xlim(0, sub['usable'].max() * 1.18)
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'plot_sector_{sec:02d}_detail.png'),
                dpi=120, bbox_inches='tight')
    plt.show()

    display(sub.rename(columns={'usable': 'Usable transits'}))

In [ ]:
# Cell 6 — Sector combination overlap
# For a chosen set of sectors, how many planets are covered and how many total transits?
# Useful for deciding the minimum sector set that covers the most planets.

COMBO_SECTORS = [75, 41]   # edit this list — try different combinations
MIN_TRANSITS_PER_PLANET = 3   # a planet counts only if it has >= this many usable transits in the combo

combo_cols = [s for s in COMBO_SECTORS if s in matrix.columns]
if not combo_cols:
    print('No valid sectors in COMBO_SECTORS.')
else:
    combo = matrix[combo_cols].sum(axis=1)  # total usable transits across chosen sectors per planet
    covered = combo[combo >= MIN_TRANSITS_PER_PLANET].sort_values(ascending=False)
    not_covered = combo[combo < MIN_TRANSITS_PER_PLANET]

    sec_str = ' + '.join(f'S{s:02d}' for s in combo_cols)
    print(f'Sector combination: {sec_str}')
    print(f'Min transits per planet threshold: {MIN_TRANSITS_PER_PLANET}')
    print(f'  Planets covered : {len(covered)}')
    print(f'  Planets missing : {len(not_covered)}')
    print(f'  Total transits  : {int(covered.sum())}')

    print(f'\nCovered planets:')
    for p, v in covered.items():
        # Show per-sector breakdown
        breakdown = '  '.join(f'S{s:02d}:{int(matrix.loc[p,s])}' for s in combo_cols)
        print(f'  {p:<22} {int(v):>3} transits   [{breakdown}]')

    if len(not_covered):
        print(f'\nNot covered (< {MIN_TRANSITS_PER_PLANET} transits):')
        for p, v in not_covered.items():
            print(f'  {p:<22} {int(v):>3} transits in chosen sectors')

In [ ]:
# Cell 7 — Distribution statistics
usable = df_summary['total_usable']

print('=== Planet usable transit statistics ===')
print(f'  Count   : {len(usable)}')
print(f'  Mean    : {usable.mean():.1f}')
print(f'  Median  : {usable.median():.1f}')
print(f'  Std     : {usable.std():.1f}')
print(f'  Min     : {usable.min()}  ({df_summary.loc[usable.idxmin(),"System"]})')
print(f'  Max     : {usable.max()}  ({df_summary.loc[usable.idxmax(),"System"]})')
print(f'  25th p  : {usable.quantile(0.25):.0f}')
print(f'  75th p  : {usable.quantile(0.75):.0f}')

print('\n=== Sector statistics ===')
print(f'  Sectors with ≥1 planet  : {(df_sectors["n_planets"] >= 1).sum()}')
print(f'  Sectors with ≥5 planets : {(df_sectors["n_planets"] >= 5).sum()}')
print(f'  Sectors with ≥10 planets: {(df_sectors["n_planets"] >= 10).sum()}')
print(f'  Max planets in a sector : {df_sectors["n_planets"].max()}  '
      f'(S{int(df_sectors.iloc[0]["sector"]):02d})')

# Histogram of usable transits per planet
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(usable, bins=15, color='#2C7BB6', edgecolor='white', linewidth=0.6)
axes[0].axvline(usable.median(), color='#D7191C', lw=1.5, ls='--',
                label=f'Median = {usable.median():.0f}')
axes[0].axvline(usable.mean(), color='#FD8D3C', lw=1.5, ls=':',
                label=f'Mean = {usable.mean():.1f}')
axes[0].set_xlabel('Total usable transits', fontsize=10)
axes[0].set_ylabel('Number of planets', fontsize=10)
axes[0].set_title('Distribution: Usable Transits per Planet', fontsize=10, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].spines[['top','right']].set_visible(False)

# Histogram of planets per sector (top 40 sectors)
top40 = df_sectors.head(40)
axes[1].bar(range(len(top40)),
            top40['n_planets'],
            color='#756BB1', edgecolor='white', linewidth=0.4)
axes[1].set_xticks(range(len(top40)))
axes[1].set_xticklabels([f'S{int(s):02d}' for s in top40['sector']],
                        rotation=90, fontsize=6.5)
axes[1].set_ylabel('Number of planets', fontsize=10)
axes[1].set_title('Planets per Sector (Top 40)', fontsize=10, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'plot_distributions.png'), dpi=130, bbox_inches='tight')
plt.show()